# 09 - Modern Robot Learning And VLAs

## What / Why / How

**What we are trying to do:** Connect fundamentals to modern robot learning: action chunks, diffusion/flow policies, action tokens, and vision-language-action models.

**Why this matters:** Current frontier systems still depend on classical robotics concepts, but they use large datasets and distributional policies to handle messy real tasks.

**How we will do it:** Use toy action distributions to see why mean regression fails, sample safe actions, tokenize actions, and inspect a minimal VLA-style episode record.

## Goal

Connect the fundamentals to current robot-learning research.

You will learn the ideas behind:

- Action chunks
- Diffusion and flow policies
- Action tokenization
- Vision-language-action models
- Embodied reasoning systems

This notebook is conceptual and lightweight. Training a real VLA requires GPUs, robot datasets, careful evaluation, and safety infrastructure.

In [ ]:
import math
import random
from collections import defaultdict

import numpy as np

try:
    import matplotlib.pyplot as plt
    HAS_PLOT = True
except ModuleNotFoundError:
    plt = None
    HAS_PLOT = False

np.set_printoptions(precision=3, suppress=True)

def plot_unavailable():
    if not HAS_PLOT:
        print("Install matplotlib to see the plot: pip install -r requirements.txt")

## Research Snapshot

Important systems and papers:

- Open X-Embodiment and RT-X: pooled robot data across embodiments.
- Diffusion Policy: action-sequence generation by denoising.
- Octo: open generalist robot policy.
- OpenVLA: open-source VLA trained on Open X-Embodiment episodes.
- pi0: VLA flow model for general robot control.
- SmolVLA and LeRobot: accessible open robot-learning stack.
- GR00T and Gemini Robotics: industry-scale foundation models, embodied reasoning, and humanoid control.

Read the project research brief after this notebook:
`research/latest_research_brief.md`

## Why Mean Regression Can Fail

In [ ]:
rng = np.random.default_rng(123)

# A robot can go around an obstacle above or below.
# Both actions are valid. Averaging them points into the obstacle.
upper_actions = np.column_stack([
    np.ones(100) * 0.8,
    rng.normal(0.55, 0.05, size=100),
])
lower_actions = np.column_stack([
    np.ones(100) * 0.8,
    rng.normal(-0.55, 0.05, size=100),
])
demo_actions = np.vstack([upper_actions, lower_actions])

mean_action = demo_actions.mean(axis=0)
sampled_actions = demo_actions[rng.choice(len(demo_actions), size=8, replace=False)]

def hits_obstacle(action):
    # Toy obstacle centered at y=0. The averaged action goes through it.
    return abs(action[1]) < 0.2 and action[0] > 0.4

print("mean action:", mean_action, "hits obstacle?", hits_obstacle(mean_action))
print("sampled actions hit obstacle?", [hits_obstacle(a) for a in sampled_actions[:5]])

In [ ]:
if HAS_PLOT:
    plt.figure(figsize=(5, 4))
    plt.scatter(demo_actions[:, 0], demo_actions[:, 1], s=10, alpha=0.4, label="demo actions")
    plt.scatter([mean_action[0]], [mean_action[1]], c="tab:red", s=100, label="MSE mean")
    circle = plt.Circle((0.8, 0.0), 0.2, color="black", alpha=0.2, label="obstacle zone")
    plt.gca().add_patch(circle)
    plt.xlabel("forward")
    plt.ylabel("sideways")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.title("Multimodal actions")
    plt.show()
else:
    plot_unavailable()

## Diffusion And Flow Intuition

Diffusion and flow policies try to model a distribution over action sequences, not just one average action.

In real manipulation, this matters because there may be many valid grasps, paths, and recovery motions.

In [ ]:
def toy_action_sampler(num_samples=10, noise=0.03):
    choices = demo_actions[rng.choice(len(demo_actions), size=num_samples)]
    return choices + rng.normal(0, noise, size=choices.shape)

samples = toy_action_sampler(6)
print("sampled candidate actions:")
print(samples)
print("choose one with a safety filter:")
safe = [a for a in samples if not hits_obstacle(a)]
print(safe[0] if safe else "no safe sample")

## Action Tokenization

In [ ]:
def tokenize_actions(actions, low=-1.0, high=1.0, bins=256):
    actions = np.clip(actions, low, high)
    scaled = (actions - low) / (high - low)
    tokens = np.round(scaled * (bins - 1)).astype(int)
    return tokens

def detokenize_actions(tokens, low=-1.0, high=1.0, bins=256):
    scaled = tokens / (bins - 1)
    return low + scaled * (high - low)

action_chunk = np.array([[0.2, -0.1, 0.0], [0.3, -0.1, 1.0], [0.25, 0.0, 1.0]])
tokens = tokenize_actions(action_chunk)
reconstructed = detokenize_actions(tokens)
print("action chunk:")
print(action_chunk)
print("tokens:")
print(tokens)
print("reconstructed:")
print(reconstructed)

## A Minimal VLA Data Record

In [ ]:
robot_episode = {
    "instruction": "pick up the red block and place it in the bowl",
    "image_embedding_shape": (16, 16, 768),
    "proprioception": {
        "joint_positions": np.zeros(7),
        "gripper_width": 0.04,
    },
    "action_chunk": action_chunk,
}

for key, value in robot_episode.items():
    print(key, "=>", value)

## Exercises

1. Add a third action mode to the multimodal example.
2. Change token bins from 256 to 16. How much precision is lost?
3. Write down what data you would need to train a robot to open a drawer.
4. Read one source from each section of `research/latest_research_brief.md`.